# 12_Economic_ Impact_Assesment_Part2

# 01. CARGA Y RECONOCIMIENTO DE FUENTES

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# NB_11 — ECONOMIC IMPACT ASSESSMENT
# 01. CARGA Y RECONOCIMIENTO DE FUENTES
# ============================================================
.
ROOT = Path(r"C:\ds4b\C2011-13 Wind Farm_v4")

PATH_MASTER = ROOT / "05_Outputs" / "Activity_Master_with_Materials.xlsx"
PATH_SCENARIOS = ROOT / "07_Outputs" / "Scenario_Simulation_Results.xlsx"
PATH_ACTIONS = ROOT / "07_Outputs" / "Corrective_Action_Evaluation.xlsx"
PATH_CONFIG = ROOT / "06_Outputs" / "06_Decision_Points_Base.xlsx"

# ------------------------------------------------------------
# 1. Comprobar existencia de archivos
# ------------------------------------------------------------

sources = {
    "Activity Master": PATH_MASTER,
    "Scenario Simulation": PATH_SCENARIOS,
    "Corrective Action Evaluation": PATH_ACTIONS,
    "Decision Points Base": PATH_CONFIG
}

print("=" * 65)
print("COMPROBACIÓN DE FUENTES")
print("=" * 65)

for name, path in sources.items():
    print(f"{name}: {'OK' if path.exists() else 'NO ENCONTRADO'}")

missing = [name for name, path in sources.items() if not path.exists()]

if missing:
    raise FileNotFoundError(f"Faltan archivos: {missing}")

# ------------------------------------------------------------
# 2. Cargar hojas relevantes
# ------------------------------------------------------------

df_master = pd.read_excel(PATH_MASTER, sheet_name="Sheet1")

df_scenario_summary = pd.read_excel(
    PATH_SCENARIOS,
    sheet_name="Scenario Summary"
)

df_activity_comparison = pd.read_excel(
    PATH_SCENARIOS,
    sheet_name="Activity Comparison"
)

df_corrective_summary = pd.read_excel(
    PATH_ACTIONS,
    sheet_name="Decision Summary"
)

df_corrective_actions = pd.read_excel(
    PATH_ACTIONS,
    sheet_name="Corrective Actions"
)

df_supply_risk = pd.read_excel(
    PATH_ACTIONS,
    sheet_name="Supply Risk"
)

# ------------------------------------------------------------
# 3. Reconocimiento de dimensiones
# ------------------------------------------------------------

datasets = {
    "Activity Master": df_master,
    "Scenario Summary": df_scenario_summary,
    "Activity Comparison": df_activity_comparison,
    "Decision Summary": df_corrective_summary,
    "Corrective Actions": df_corrective_actions,
    "Supply Risk": df_supply_risk
}

print("\n" + "=" * 65)
print("DIMENSIONES Y COLUMNAS")
print("=" * 65)

for name, df in datasets.items():
    print(f"\n{name}: {df.shape}")
    print(list(df.columns))

# 02. DIAGNÓSTICO ECONÓMICO Y TRAZABILIDAD

In [2]:
# ============================================================
# 02. DIAGNÓSTICO ECONÓMICO Y TRAZABILIDAD
# ============================================================

# ------------------------------------------------------------
# 1. Presupuesto del proyecto: solo actividades terminales
# ------------------------------------------------------------

# Evitamos duplicidades de presupuesto en niveles WBS superiores.
# El catálogo contiene 135 actividades terminales.
# Las actividades afectadas son también terminales.

terminal_ids = set(
    df_activity_comparison["ID"].dropna().unique()
)

df_terminal = df_master[
    df_master["ID"].isin(terminal_ids)
].copy()

budget_summary = pd.DataFrame({
    "Concepto": [
        "Presupuesto total",
        "Costes fijos",
        "Costes variables",
        "Costes de recursos"
    ],
    "Importe_EUR": [
        df_terminal["Total_Cost_Plan"].sum(),
        df_terminal["Fixed_Cost"].sum(),
        df_terminal["Variable_Cost"].sum(),
        df_terminal["Resource_Cost"].sum()
    ]
})

print("=" * 75)
print("1. PRESUPUESTO DE REFERENCIA")
print("=" * 75)

display(budget_summary)

print(
    "\nReconciliación costes:",
    round(
        df_terminal[
            ["Fixed_Cost", "Variable_Cost", "Resource_Cost"]
        ].sum().sum()
        - df_terminal["Total_Cost_Plan"].sum(),
        2
    ),
    "EUR"
)

# ------------------------------------------------------------
# 2. Ficha económica de las actividades afectadas
# ------------------------------------------------------------

target_ids = [20, 79]

df_target_activities = df_master[
    df_master["ID"].isin(target_ids)
][[
    "ID",
    "Name",
    "Duration_Plan_Days",
    "Total_Cost_Plan",
    "Fixed_Cost",
    "Variable_Cost",
    "Resource_Cost",
    "Supply_Delay_Days",
    "Buffer_Days",
    "Material_Status"
]].copy()

print("\n" + "=" * 75)
print("2. ACTIVIDADES AFECTADAS")
print("=" * 75)

display(
    df_target_activities.sort_values("ID")
)

# ------------------------------------------------------------
# 3. Escenarios antes y después de las acciones
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("3. RESULTADOS DE LOS ESCENARIOS")
print("=" * 75)

display(
    df_corrective_summary.sort_values("Scenario_ID")
)

# ------------------------------------------------------------
# 4. Acciones correctivas y recuperación
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("4. ACCIONES CORRECTIVAS")
print("=" * 75)

display(
    df_corrective_actions.sort_values("Action_ID")
)

# ------------------------------------------------------------
# 5. Riesgo residual de suministro
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("5. RIESGO RESIDUAL DE SUMINISTRO")
print("=" * 75)

display(
    df_supply_risk.sort_values("Scenario_ID")
)

# ------------------------------------------------------------
# 6. Comprobación de costes y datos económicos disponibles
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("6. DISPONIBILIDAD DE INFORMACIÓN ECONÓMICA")
print("=" * 75)

economic_columns = [
    "Total_Cost_Plan",
    "Fixed_Cost",
    "Variable_Cost",
    "Resource_Cost"
]

print("\nValores nulos:")
display(df_master[economic_columns].isna().sum().to_frame("Nulos"))

print("\nValores cero:")
display(
    (df_master[economic_columns] == 0)
    .sum()
    .to_frame("Registros con valor cero")
)

print("\nCostes de recursos distintos de cero:")
display(
    df_master.loc[
        df_master["Resource_Cost"].fillna(0) != 0,
        ["ID", "Name", "Resource_Cost"]
    ]
)

print("\nDiagnóstico terminado.")

# 03. REGISTRO DE HIPÓTESIS ECONÓMICAS

In [3]:
# ============================================================
# 03. REGISTRO DE HIPÓTESIS ECONÓMICAS
# ============================================================

# ------------------------------------------------------------
# 1. Datos conocidos del simulador
# ------------------------------------------------------------

known_facts = pd.DataFrame([
    {
        "Indicador": "CA01 - Reducción del impacto de suministro",
        "Valor": 10,
        "Unidad": "días",
        "Origen": "Corrective Action Evaluation"
    },
    {
        "Indicador": "CA01 - Reducción del exceso sobre buffer",
        "Valor": 10,
        "Unidad": "días",
        "Origen": "Supply Risk"
    },
    {
        "Indicador": "CA02 - Recuperación de duración del proyecto",
        "Valor": 5,
        "Unidad": "días",
        "Origen": "Decision Summary"
    },
    {
        "Indicador": "Buffer de suministro actividad 79",
        "Valor": 21,
        "Unidad": "días",
        "Origen": "Activity Master"
    },
    {
        "Indicador": "Exceso residual sobre buffer CA01",
        "Valor": 19,
        "Unidad": "días",
        "Origen": "Supply Risk"
    }
])

print("=" * 75)
print("DATOS CONOCIDOS — NO SON HIPÓTESIS")
print("=" * 75)

display(known_facts)

# ------------------------------------------------------------
# 2. Parámetros económicos editables
# ------------------------------------------------------------

assumption_definitions = [
    # CA01 - Coste de intervención
    ("CA01", "urgent_transport_premium",
     "Sobrecoste de transporte urgente", "EUR",
     "Coste incremental frente al transporte ordinario"),

    ("CA01", "supplier_expedite_fee",
     "Recargo del proveedor por acelerar el suministro", "EUR",
     "Cargo adicional específico por priorización"),

    ("CA01", "extra_handling_coordination",
     "Manipulación y coordinación extraordinarias", "EUR",
     "Costes adicionales no incluidos en el transporte"),

    # CA01 - Coste esperado de no intervenir
    ("CA01", "supply_disruption_probability",
     "Probabilidad de que el retraso genere una incidencia operativa",
     "0-1",
     "Probabilidad estimada, no porcentaje del presupuesto"),

    ("CA01", "supply_disruption_cost_per_day",
     "Consecuencia económica diaria de la incidencia de suministro",
     "EUR/día",
     "Coste incremental atribuible a cada día de incidencia"),

    # CA02 - Coste de intervención
    ("CA02", "overtime_hours",
     "Horas extraordinarias adicionales", "horas",
     "Horas realmente imputables a la recuperación"),

    ("CA02", "overtime_incremental_cost_per_hour",
     "Coste incremental por hora extraordinaria", "EUR/hora",
     "Coste adicional respecto al coste ordinario"),

    ("CA02", "subcontractor_extra_days",
     "Días adicionales de refuerzo externo", "días",
     "Días de contratación extraordinaria"),

    ("CA02", "subcontractor_incremental_cost_per_day",
     "Coste incremental diario del refuerzo externo", "EUR/día",
     "Tarifa adicional del recurso externo"),

    ("CA02", "equipment_extra_days",
     "Días adicionales de maquinaria o equipos", "días",
     "Uso extraordinario necesario para recuperar plazo"),

    ("CA02", "equipment_incremental_cost_per_day",
     "Coste incremental diario de maquinaria", "EUR/día",
     "Alquiler o disponibilidad adicional"),

    # CA02 - Coste esperado de no intervenir
    ("CA02", "avoidable_project_overhead_per_day",
     "Coste diario de estructura de obra potencialmente evitable",
     "EUR/día",
     "Solo costes que desaparecen realmente al acortar el proyecto")
]

scenario_columns = [
    "Pessimistic",
    "Realistic",
    "Optimistic"
]

df_economic_assumptions = pd.DataFrame([
    {
        "Action_ID": action,
        "Parameter": parameter,
        "Description": description,
        "Unit": unit,
        "Economic_Definition": definition,
        **{scenario: np.nan for scenario in scenario_columns}
    }
    for action, parameter, description, unit, definition
    in assumption_definitions
])

print("\n" + "=" * 75)
print("REGISTRO DE HIPÓTESIS ECONÓMICAS EDITABLES")
print("=" * 75)

display(df_economic_assumptions)

# ------------------------------------------------------------
# 3. Reglas de interpretación
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("REGLAS DEL MODELO")
print("=" * 75)

rules = [
    "CA01: el riesgo evitado se calcula sobre los 10 días de reducción.",
    "CA01: no se monetiza como retraso del proyecto.",
    "CA02: el ahorro temporal se calcula sobre los 5 días recuperados.",
    "CA02: solo se imputan costes diarios de estructura realmente evitables.",
    "Los costes de intervención son incrementales, no costes ordinarios.",
    "No se utiliza el presupuesto de actividad como coste diario.",
    "Los tres escenarios tendrán hipótesis independientes y editables."
]

for rule in rules:
    print(f"• {rule}")

print("\nParámetros creados:", len(df_economic_assumptions))
print("Valores económicos pendientes de parametrización.")

# 04. ESTRUCTURA DEL MODELO ECONÓMICO

In [4]:
# ============================================================
# 04. ESTRUCTURA DEL MODELO ECONÓMICO
# ============================================================

# ------------------------------------------------------------
# 1. Componentes de coste de intervención
# ------------------------------------------------------------

intervention_components = pd.DataFrame([
    {
        "Action_ID": "CA01",
        "Component": "Transporte urgente",
        "Parameter": "urgent_transport_premium"
    },
    {
        "Action_ID": "CA01",
        "Component": "Recargo proveedor",
        "Parameter": "supplier_expedite_fee"
    },
    {
        "Action_ID": "CA01",
        "Component": "Manipulación y coordinación",
        "Parameter": "extra_handling_coordination"
    },
    {
        "Action_ID": "CA02",
        "Component": "Horas extraordinarias",
        "Parameter": "overtime_hours × overtime_incremental_cost_per_hour"
    },
    {
        "Action_ID": "CA02",
        "Component": "Refuerzo externo",
        "Parameter": "subcontractor_extra_days × subcontractor_incremental_cost_per_day"
    },
    {
        "Action_ID": "CA02",
        "Component": "Maquinaria adicional",
        "Parameter": "equipment_extra_days × equipment_incremental_cost_per_day"
    }
])

# ------------------------------------------------------------
# 2. Componentes de coste de no intervenir
# ------------------------------------------------------------

non_intervention_components = pd.DataFrame([
    {
        "Action_ID": "CA01",
        "Component": "Riesgo de incidencia de suministro",
        "Formula": (
            "10 días de exposición reducida × "
            "probabilidad de incidencia × coste diario de incidencia"
        )
    },
    {
        "Action_ID": "CA02",
        "Component": "Estructura de obra evitable",
        "Formula": (
            "5 días recuperados × coste diario de estructura evitable"
        )
    }
])

print("=" * 75)
print("COMPONENTES DEL COSTE DE INTERVENCIÓN")
print("=" * 75)

display(intervention_components)

print("\n" + "=" * 75)
print("COMPONENTES DEL COSTE DE NO INTERVENIR")
print("=" * 75)

display(non_intervention_components)

# ------------------------------------------------------------
# 3. Validación de independencia económica
# ------------------------------------------------------------

validation_rules = pd.DataFrame([
    {
        "Control": "CA01 no genera ahorro de duración del proyecto",
        "Validación": "Project_Days_Recovered = 0 en SC01"
    },
    {
        "Control": "CA02 recupera cinco días de proyecto",
        "Validación": "Project_Days_Recovered = 5 en SC02 y SC03"
    },
    {
        "Control": "CA01 y CA02 no duplican el ahorro temporal",
        "Validación": "CA01 se valora como riesgo de suministro; CA02 como estructura"
    },
    {
        "Control": "Costes incrementales",
        "Validación": "Se excluyen costes ordinarios ya presupuestados"
    },
    {
        "Control": "Beneficios evitables",
        "Validación": "No se monetizan costes comprometidos o no recuperables"
    }
])

print("\n" + "=" * 75)
print("CONTROLES METODOLÓGICOS")
print("=" * 75)

display(validation_rules)

**Parametrización de escenarios**

| Parámetro | Pesimista | Realista | Optimista |
| :--- | :--- | :--- | :--- |
| **Transporte urgente (€)** | 5.000 | 3.500 | 2.500 |
| **Recargo proveedor (€)** | 3.000 | 2.000 | 1.000 |
| **Manipulación (€)** | 1.500 | 1.000 | 500 |
| **Probabilidad incidencia CA01** | 35% | 55% | 75% |
| **Coste incidencia CA01 (€/día)** | 1.000 | 2.000 | 3.000 |
| **Horas extra CA02** | 120 | 90 | 60 |
| **Coste incremental hora extra (€)** | 35 | 30 | 25 |
| **Días refuerzo externo** | 5 | 3 | 2 |
| **Coste incremental refuerzo (€/día)** | 1.500 | 1.200 | 1.000 |
| **Días maquinaria adicional** | 5 | 3 | 2 |
| **Coste incremental maquinaria (€/día)** | 800 | 600 | 400 |
| **Estructura evitable (€/día)** | 2.000 | 3.500 | 5.000 |

# 05. PARAMETRIZACIÓN DE ESCENARIOS ECONÓMICOS

In [6]:
# ============================================================
# 05. PARAMETRIZACIÓN DE ESCENARIOS ECONÓMICOS
# ============================================================

scenario_values = {
    "Pessimistic": {
        "urgent_transport_premium": 5000,
        "supplier_expedite_fee": 3000,
        "extra_handling_coordination": 1500,
        "supply_disruption_probability": 0.35,
        "supply_disruption_cost_per_day": 1000,
        "overtime_hours": 120,
        "overtime_incremental_cost_per_hour": 35,
        "subcontractor_extra_days": 5,
        "subcontractor_incremental_cost_per_day": 1500,
        "equipment_extra_days": 5,
        "equipment_incremental_cost_per_day": 800,
        "avoidable_project_overhead_per_day": 2000
    },
    "Realistic": {
        "urgent_transport_premium": 3500,
        "supplier_expedite_fee": 2000,
        "extra_handling_coordination": 1000,
        "supply_disruption_probability": 0.55,
        "supply_disruption_cost_per_day": 2000,
        "overtime_hours": 90,
        "overtime_incremental_cost_per_hour": 30,
        "subcontractor_extra_days": 3,
        "subcontractor_incremental_cost_per_day": 1200,
        "equipment_extra_days": 3,
        "equipment_incremental_cost_per_day": 600,
        "avoidable_project_overhead_per_day": 3500
    },
    "Optimistic": {
        "urgent_transport_premium": 2500,
        "supplier_expedite_fee": 1000,
        "extra_handling_coordination": 500,
        "supply_disruption_probability": 0.75,
        "supply_disruption_cost_per_day": 3000,
        "overtime_hours": 60,
        "overtime_incremental_cost_per_hour": 25,
        "subcontractor_extra_days": 2,
        "subcontractor_incremental_cost_per_day": 1000,
        "equipment_extra_days": 2,
        "equipment_incremental_cost_per_day": 400,
        "avoidable_project_overhead_per_day": 5000
    }
}

# Actualizar la tabla de hipótesis existente
for scenario, values in scenario_values.items():
    df_economic_assumptions[scenario] = (
        df_economic_assumptions["Parameter"]
        .map(values)
    )

print("=" * 75)
print("HIPÓTESIS ECONÓMICAS POR ESCENARIO")
print("=" * 75)

display(df_economic_assumptions)

# Comprobación de integridad
assert df_economic_assumptions[scenario_columns].notna().all().all()
assert (
    df_economic_assumptions.loc[
        df_economic_assumptions["Parameter"] == "supply_disruption_probability",
        scenario_columns
    ].values <= 1
).all()

print("\nValidación completada.")
print("Parámetros:", len(df_economic_assumptions))
print("Escenarios:", len(scenario_columns))

# 06. EVALUACIÓN ECONÓMICA DE LAS ACCIONES

In [7]:
# ============================================================
# 06. EVALUACIÓN ECONÓMICA DE LAS ACCIONES
# ============================================================

results = []

for scenario in scenario_columns:

    params = scenario_values[scenario]

    # --------------------------------------------------------
    # CA01 - SUPPLIER EXPEDITING
    # --------------------------------------------------------

    ca01 = df_corrective_actions[
        df_corrective_actions["Action_ID"] == "CA01"
    ].iloc[0]

    days_before = float(ca01["Scenario_Impact_Before_Days"])
    days_after = float(ca01["Scenario_Impact_After_Days"])

    probability = params["supply_disruption_probability"]
    daily_impact_cost = params["supply_disruption_cost_per_day"]

    ca01_intervention_cost = (
        params["urgent_transport_premium"]
        + params["supplier_expedite_fee"]
        + params["extra_handling_coordination"]
    )

    ca01_cost_without = (
        days_before * probability * daily_impact_cost
    )

    ca01_residual_cost = (
        days_after * probability * daily_impact_cost
    )

    ca01_avoided_cost = (
        ca01_cost_without - ca01_residual_cost
    )

    ca01_net_benefit = (
        ca01_avoided_cost - ca01_intervention_cost
    )

    ca01_roi = (
        ca01_net_benefit / ca01_intervention_cost * 100
        if ca01_intervention_cost > 0 else np.nan
    )

    results.append({
        "Scenario": scenario,
        "Action_ID": "CA01",
        "Action": "Supplier Expediting",
        "Impact_Before_Days": days_before,
        "Impact_After_Days": days_after,
        "Intervention_Cost_EUR": ca01_intervention_cost,
        "Cost_Without_Action_EUR": ca01_cost_without,
        "Residual_Cost_EUR": ca01_residual_cost,
        "Avoided_Cost_EUR": ca01_avoided_cost,
        "Net_Benefit_EUR": ca01_net_benefit,
        "ROI_Pct": ca01_roi,
        "Break_Even_Intervention_EUR": ca01_avoided_cost
    })

    # --------------------------------------------------------
    # CA02 - EXECUTION RECOVERY
    # --------------------------------------------------------

    ca02 = df_corrective_actions[
        df_corrective_actions["Action_ID"] == "CA02"
    ].iloc[0]

    days_before = float(ca02["Scenario_Impact_Before_Days"])
    days_after = float(ca02["Scenario_Impact_After_Days"])

    overhead_daily = params["avoidable_project_overhead_per_day"]

    ca02_intervention_cost = (
        params["overtime_hours"]
        * params["overtime_incremental_cost_per_hour"]
        +
        params["subcontractor_extra_days"]
        * params["subcontractor_incremental_cost_per_day"]
        +
        params["equipment_extra_days"]
        * params["equipment_incremental_cost_per_day"]
    )

    ca02_cost_without = days_before * overhead_daily
    ca02_residual_cost = days_after * overhead_daily

    ca02_avoided_cost = (
        ca02_cost_without - ca02_residual_cost
    )

    ca02_net_benefit = (
        ca02_avoided_cost - ca02_intervention_cost
    )

    ca02_roi = (
        ca02_net_benefit / ca02_intervention_cost * 100
        if ca02_intervention_cost > 0 else np.nan
    )

    results.append({
        "Scenario": scenario,
        "Action_ID": "CA02",
        "Action": "Execution Recovery",
        "Impact_Before_Days": days_before,
        "Impact_After_Days": days_after,
        "Intervention_Cost_EUR": ca02_intervention_cost,
        "Cost_Without_Action_EUR": ca02_cost_without,
        "Residual_Cost_EUR": ca02_residual_cost,
        "Avoided_Cost_EUR": ca02_avoided_cost,
        "Net_Benefit_EUR": ca02_net_benefit,
        "ROI_Pct": ca02_roi,
        "Break_Even_Intervention_EUR": ca02_avoided_cost
    })


# ------------------------------------------------------------
# 3. Tabla de resultados
# ------------------------------------------------------------

df_economic_results = pd.DataFrame(results)

money_columns = [
    "Intervention_Cost_EUR",
    "Cost_Without_Action_EUR",
    "Residual_Cost_EUR",
    "Avoided_Cost_EUR",
    "Net_Benefit_EUR",
    "Break_Even_Intervention_EUR"
]

df_economic_results[money_columns] = (
    df_economic_results[money_columns].round(2)
)

df_economic_results["ROI_Pct"] = (
    df_economic_results["ROI_Pct"].round(2)
)

print("=" * 100)
print("RESULTADOS ECONÓMICOS POR ACCIÓN Y ESCENARIO")
print("=" * 100)

display(
    df_economic_results.sort_values(
        ["Action_ID", "Scenario"]
    ).reset_index(drop=True)
)

# ------------------------------------------------------------
# 4. Diagnóstico de rentabilidad
# ------------------------------------------------------------

df_economic_results["Economic_Decision"] = np.select(
    [
        df_economic_results["Net_Benefit_EUR"] > 0,
        df_economic_results["Net_Benefit_EUR"] < 0
    ],
    [
        "ECONOMICALLY VIABLE",
        "NOT ECONOMICALLY VIABLE"
    ],
    default="BREAK-EVEN"
)

print("\n" + "=" * 100)
print("DIAGNÓSTICO DE RENTABILIDAD")
print("=" * 100)

display(
    df_economic_results[
        [
            "Scenario",
            "Action_ID",
            "Intervention_Cost_EUR",
            "Avoided_Cost_EUR",
            "Net_Benefit_EUR",
            "ROI_Pct",
            "Economic_Decision"
        ]
    ].sort_values(["Action_ID", "Scenario"])
)

# 07. EVALUACIÓN ECONÓMICA CONJUNT

In [8]:
# ============================================================
# 07. EVALUACIÓN ECONÓMICA CONJUNTA — SC03
# ============================================================

df_combined = (
    df_economic_results
    .groupby("Scenario", as_index=False)
    .agg(
        Intervention_Cost_EUR=("Intervention_Cost_EUR", "sum"),
        Avoided_Cost_EUR=("Avoided_Cost_EUR", "sum"),
        Net_Benefit_EUR=("Net_Benefit_EUR", "sum")
    )
)

df_combined["ROI_Pct"] = (
    df_combined["Net_Benefit_EUR"]
    / df_combined["Intervention_Cost_EUR"] * 100
)

df_combined["Break_Even_Intervention_EUR"] = (
    df_combined["Avoided_Cost_EUR"]
)

df_combined["Economic_Decision"] = np.select(
    [
        df_combined["Net_Benefit_EUR"] > 0,
        df_combined["Net_Benefit_EUR"] < 0
    ],
    [
        "ECONOMICALLY VIABLE",
        "NOT ECONOMICALLY VIABLE"
    ],
    default="BREAK-EVEN"
)

df_combined = df_combined.round(2)

print("=" * 90)
print("SC03 — EVALUACIÓN ECONÓMICA CONJUNTA")
print("=" * 90)

display(
    df_combined.sort_values("Scenario")
)

# ------------------------------------------------------------
# Comprobación de consistencia con las acciones individuales
# ------------------------------------------------------------

for scenario in scenario_columns:

    individual = df_economic_results[
        df_economic_results["Scenario"] == scenario
    ]

    combined = df_combined[
        df_combined["Scenario"] == scenario
    ].iloc[0]

    assert np.isclose(
        individual["Intervention_Cost_EUR"].sum(),
        combined["Intervention_Cost_EUR"]
    )

    assert np.isclose(
        individual["Avoided_Cost_EUR"].sum(),
        combined["Avoided_Cost_EUR"]
    )

    assert np.isclose(
        individual["Net_Benefit_EUR"].sum(),
        combined["Net_Benefit_EUR"]
    )

print("\nValidación: agregación conjunta sin diferencias.")

# 08. UMBRALES DE RENTABILIDAD Y MATRIZ DE DECISIÓN

In [9]:
# ============================================================
# 08. UMBRALES DE RENTABILIDAD Y MATRIZ DE DECISIÓN
# ============================================================

thresholds = []

for scenario in scenario_columns:

    p = scenario_values[scenario]

    for action_id in ["CA01", "CA02"]:

        result = df_economic_results[
            (df_economic_results["Scenario"] == scenario) &
            (df_economic_results["Action_ID"] == action_id)
        ].iloc[0]

        if action_id == "CA01":

            probability = p["supply_disruption_probability"]
            avoided_days = 10

            # Coste diario mínimo de incidencia para alcanzar break-even
            minimum_daily_cost = (
                result["Intervention_Cost_EUR"]
                / (avoided_days * probability)
            )

            thresholds.append({
                "Scenario": scenario,
                "Action_ID": action_id,
                "Maximum_Intervention_Cost_EUR":
                    result["Avoided_Cost_EUR"],
                "Minimum_Daily_Impact_Cost_EUR":
                    minimum_daily_cost,
                "Minimum_Avoidable_Days": np.nan,
                "Net_Benefit_EUR": result["Net_Benefit_EUR"],
                "ROI_Pct": result["ROI_Pct"]
            })

        else:

            avoided_days = 5

            # Coste diario mínimo de estructura evitable
            minimum_daily_cost = (
                result["Intervention_Cost_EUR"]
                / avoided_days
            )

            thresholds.append({
                "Scenario": scenario,
                "Action_ID": action_id,
                "Maximum_Intervention_Cost_EUR":
                    result["Avoided_Cost_EUR"],
                "Minimum_Daily_Impact_Cost_EUR":
                    minimum_daily_cost,
                "Minimum_Avoidable_Days": (
                    result["Intervention_Cost_EUR"]
                    / p["avoidable_project_overhead_per_day"]
                ),
                "Net_Benefit_EUR": result["Net_Benefit_EUR"],
                "ROI_Pct": result["ROI_Pct"]
            })

df_break_even = pd.DataFrame(thresholds).round(2)

print("=" * 90)
print("UMBRALES DE RENTABILIDAD POR ACCIÓN")
print("=" * 90)

display(df_break_even)

# ------------------------------------------------------------
# Matriz de decisión económica
# ------------------------------------------------------------

df_decision_matrix = df_economic_results[
    [
        "Scenario",
        "Action_ID",
        "Intervention_Cost_EUR",
        "Avoided_Cost_EUR",
        "Net_Benefit_EUR",
        "ROI_Pct",
        "Economic_Decision"
    ]
].copy()

df_decision_matrix["Decision"] = np.select(
    [
        df_decision_matrix["Net_Benefit_EUR"] > 0,
        df_decision_matrix["Net_Benefit_EUR"] == 0
    ],
    [
        "GO - SUBJECT TO RISK REVIEW",
        "BREAK-EVEN - MANAGEMENT REVIEW"
    ],
    default="NO-GO - RENEGOTIATE OR MITIGATE"
)

# Riesgo residual de suministro
df_decision_matrix["Residual_Supply_Risk"] = np.where(
    df_decision_matrix["Action_ID"] == "CA01",
    "19 days above buffer",
    "Not applicable"
)

# Riesgo de plazo residual
df_decision_matrix["Residual_Project_Delay_Days"] = np.where(
    df_decision_matrix["Action_ID"] == "CA02",
    2,
    0
)

print("\n" + "=" * 90)
print("MATRIZ DE DECISIÓN")
print("=" * 90)

display(
    df_decision_matrix.sort_values(
        ["Action_ID", "Scenario"]
    ).reset_index(drop=True)
)

# ------------------------------------------------------------
# Resultado conjunto SC03
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("DECISIÓN CONJUNTA SC03")
print("=" * 90)

display(
    df_combined[
        [
            "Scenario",
            "Intervention_Cost_EUR",
            "Avoided_Cost_EUR",
            "Net_Benefit_EUR",
            "ROI_Pct",
            "Economic_Decision"
        ]
    ].sort_values("Scenario")
)

Lectura de los umbrales
- CA01 realista: el coste de incidencia tendría que superar 1.181,82 €/día para justificar la intervención de 6.500 €, manteniendo la probabilidad del 55 %.
- CA02 realista: bastaría con evitar 2,31 días de estructura para cubrir los 8.100 € de intervención. Tenemos una recuperación simulada de cinco días.
- CA01 pesimista: el coste de intervención supera el beneficio potencial en 6.000 €.
- CA02 pesimista: necesitaríamos evitar 7,85 días de estructura, pero solo recuperamos cinco. No compensa con esos parámetros.
Hay otra lectura relevante: CA01 puede ser económicamente rentable y, aun así, dejar un riesgo de suministro importante. La decisión económica no sustituye a la decisión operativa.

# 09. VALIDACIÓN FINAL Y RESUMEN EJECUTIVO

In [12]:
# ============================================================
# 09. VALIDACIÓN FINAL Y RESUMEN EJECUTIVO
# ============================================================

# ------------------------------------------------------------
# 1. Validación de resultados económicos
# ------------------------------------------------------------

assert len(df_economic_results) == 6
assert len(df_combined) == 3
assert len(df_break_even) == 6

assert np.allclose(
    df_economic_results["Net_Benefit_EUR"],
    df_economic_results["Avoided_Cost_EUR"]
    - df_economic_results["Intervention_Cost_EUR"]
)

assert np.allclose(
    df_economic_results["ROI_Pct"].round(2),
    (
        df_economic_results["Net_Benefit_EUR"]
        / df_economic_results["Intervention_Cost_EUR"]
        * 100
    ).round(2)
)

# ------------------------------------------------------------
# 2. Incorporar resultados operativos de SC03
# ------------------------------------------------------------

sc03 = df_corrective_summary[
    df_corrective_summary["Scenario_ID"] == "SC03"
].iloc[0]

# ------------------------------------------------------------
# 3. Resumen ejecutivo
# ------------------------------------------------------------

executive_rows = []

for scenario in scenario_columns:

    combined = df_combined[
        df_combined["Scenario"] == scenario
    ].iloc[0]

    ca01 = df_economic_results[
        (df_economic_results["Scenario"] == scenario)
        & (df_economic_results["Action_ID"] == "CA01")
    ].iloc[0]

    ca02 = df_economic_results[
        (df_economic_results["Scenario"] == scenario)
        & (df_economic_results["Action_ID"] == "CA02")
    ].iloc[0]

    executive_rows.append({
        "Scenario": scenario,
        "CA01_Net_Benefit_EUR": ca01["Net_Benefit_EUR"],
        "CA01_ROI_Pct": ca01["ROI_Pct"],
        "CA02_Net_Benefit_EUR": ca02["Net_Benefit_EUR"],
        "CA02_ROI_Pct": ca02["ROI_Pct"],
        "Combined_Intervention_Cost_EUR":
            combined["Intervention_Cost_EUR"],
        "Combined_Avoided_Cost_EUR":
            combined["Avoided_Cost_EUR"],
        "Combined_Net_Benefit_EUR":
            combined["Net_Benefit_EUR"],
        "Combined_ROI_Pct": combined["ROI_Pct"],
        "Project_Days_Recovered":
            sc03["Project_Days_Recovered"],
        "Residual_Project_Delay_Days":
            sc03["Delay_After_Actions"],
        "Residual_Supply_Impact_Days":
            sc03["Supply_Impact_After"]
    })

df_executive_summary = pd.DataFrame(executive_rows).round(2)

print("=" * 100)
print("RESUMEN EJECUTIVO — EVALUACIÓN ECONÓMICA")
print("=" * 100)

display(df_executive_summary)

# ------------------------------------------------------------
# 4. Validación de coherencia operativa
# ------------------------------------------------------------

assert sc03["Project_Days_Recovered"] == 5
assert sc03["Delay_After_Actions"] == 2
assert sc03["Supply_Impact_After"] == 5

print("\n" + "=" * 100)
print("CONTROLES FINALES")
print("=" * 100)

print("Resultados individuales:", len(df_economic_results))
print("Resultados conjuntos:", len(df_combined))
print("Umbrales calculados:", len(df_break_even))
print("Escenarios evaluados:", len(df_executive_summary))
print("Recuperación de proyecto en SC03:", sc03["Project_Days_Recovered"], "días")
print("Retraso residual del proyecto:", sc03["Delay_After_Actions"], "días")
print("Impacto residual adicional de suministro:", sc03["Supply_Impact_After"], "días")
print("\nVALIDACIÓN FINAL SUPERADA")

# 10. EXPORTACIÓN DEL INFORME ECONÓMICO

In [13]:
# ============================================================
# 10. EXPORTACIÓN DEL INFORME ECONÓMICO
# ============================================================

from pathlib import Path
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

# ------------------------------------------------------------
# 1. Ruta de salida
# ------------------------------------------------------------

output_dir = Path(
    r"C:\ds4b\C2011-13 Wind Farm_v4\07_Outputs"
)

output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "Economic_Impact_Assessment.xlsx"

# ------------------------------------------------------------
# 2. Metodología y limitaciones
# ------------------------------------------------------------

df_methodology = pd.DataFrame({
    "Aspect": [
        "Nature of assessment",
        "Economic assumptions",
        "CA01 - Supplier Expediting",
        "CA02 - Execution Recovery",
        "Combined SC03",
        "Supply risk",
        "Project delay",
        "Interpretation"
    ],
    "Description": [
        "Illustrative economic assessment based on simulated scenarios.",
        "All economic values are assumptions, not actual supplier quotations or audited costs.",
        "Expected avoided cost = reduction in exposure days × probability × daily incident cost.",
        "Avoided overhead = recovered project days × genuinely avoidable daily overhead.",
        "Individual benefits are added assuming no interaction or double counting between actions.",
        "CA01 reduces the additional scenario impact but does not eliminate the baseline supply delay.",
        "CA02 recovers 5 days, leaving 2 days of residual project delay in SC03.",
        "Results are conditional estimates, not proven savings or definitive investment decisions."
    ]
})

# ------------------------------------------------------------
# 3. Dataframes que formarán el informe
# ------------------------------------------------------------

sheets = {
    "Executive Summary": df_executive_summary,
    "Assumptions": df_economic_assumptions,
    "Individual Results": df_economic_results,
    "Combined SC03": df_combined,
    "Break-even": df_break_even,
    "Decision Matrix": df_decision_matrix,
    "Known Facts": known_facts,
    "Methodology": df_methodology
}

# ------------------------------------------------------------
# 4. Exportación
# ------------------------------------------------------------

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    for sheet_name, dataframe in sheets.items():
        dataframe.to_excel(
            writer,
            sheet_name=sheet_name,
            index=False
        )

    workbook = writer.book

    # --------------------------------------------------------
    # 5. Formato general
    # --------------------------------------------------------

    header_fill = PatternFill(
        fill_type="solid",
        fgColor="1F4E78"
    )

    header_font = Font(
        bold=True,
        color="FFFFFF"
    )

    for worksheet in workbook.worksheets:

        worksheet.freeze_panes = "A2"
        worksheet.auto_filter.ref = worksheet.dimensions

        for cell in worksheet[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(
                horizontal="center",
                vertical="center",
                wrap_text=True
            )

        worksheet.row_dimensions[1].height = 30

        for column_cells in worksheet.columns:

            max_length = max(
                len(str(cell.value))
                if cell.value is not None else 0
                for cell in column_cells
            )

            column_letter = get_column_letter(
                column_cells[0].column
            )

            worksheet.column_dimensions[column_letter].width = min(
                max(max_length + 2, 12),
                55
            )

# ------------------------------------------------------------
# 6. Comprobación del archivo
# ------------------------------------------------------------

assert output_file.exists()
assert output_file.stat().st_size > 0

check_workbook = pd.ExcelFile(output_file)

expected_sheets = list(sheets.keys())

assert check_workbook.sheet_names == expected_sheets

print("=" * 65)
print("INFORME ECONÓMICO EXPORTADO CORRECTAMENTE")
print("=" * 65)

print(f"Archivo: {output_file}")
print(f"Tamaño: {output_file.stat().st_size / 1024:.2f} KB")

print("\nHojas incluidas:")

for sheet in check_workbook.sheet_names:
    print(f" - {sheet}")

print("\nVALIDACIÓN DE EXPORTACIÓN SUPERADA")

In [14]:
# ============================================================
# AUDITORÍA WBS — 01. INVENTARIO INICIAL
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. Comprobar estructura
# ------------------------------------------------------------

required_columns = [
    "ID",
    "Name",
    "WBS",
    "Duration_Plan_Days"
]

missing_columns = [
    c for c in required_columns
    if c not in df_master.columns
]

assert not missing_columns, (
    f"Faltan columnas necesarias: {missing_columns}"
)

df_wbs_audit = df_master.copy()

# ------------------------------------------------------------
# 2. Inventario general
# ------------------------------------------------------------

total_records = len(df_wbs_audit)
unique_ids = df_wbs_audit["ID"].nunique()
duplicate_ids = df_wbs_audit["ID"].duplicated().sum()
missing_ids = df_wbs_audit["ID"].isna().sum()
missing_names = df_wbs_audit["Name"].isna().sum()
missing_wbs = df_wbs_audit["WBS"].isna().sum()

zero_duration = (
    pd.to_numeric(
        df_wbs_audit["Duration_Plan_Days"],
        errors="coerce"
    ) == 0
).sum()

missing_duration = pd.to_numeric(
    df_wbs_audit["Duration_Plan_Days"],
    errors="coerce"
).isna().sum()

# ------------------------------------------------------------
# 3. Detectar nodos estructurales y actividades terminales
# ------------------------------------------------------------

df_wbs_audit["_wbs_clean"] = (
    df_wbs_audit["WBS"].astype(str).str.strip()
)

wbs_values = set(
    df_wbs_audit["_wbs_clean"].dropna()
)

# Un registro es padre si otro registro tiene su WBS
# como prefijo seguido del separador correspondiente.
def has_children(wbs, all_wbs):
    prefix = str(wbs).rstrip(".") + "."
    return any(
        other.startswith(prefix) and other != str(wbs)
        for other in all_wbs
    )

df_wbs_audit["Has_Children"] = (
    df_wbs_audit["_wbs_clean"].apply(
        lambda x: has_children(x, wbs_values)
    )
)

df_wbs_audit["Is_Terminal"] = (
    ~df_wbs_audit["Has_Children"]
)

# ------------------------------------------------------------
# 4. Tabla de inventario
# ------------------------------------------------------------

df_wbs_inventory = pd.DataFrame({
    "Indicador": [
        "Total de registros",
        "IDs únicos",
        "IDs duplicados",
        "IDs vacíos",
        "Nombres vacíos",
        "WBS vacíos",
        "Duración cero",
        "Duración vacía",
        "Nodos con descendientes",
        "Registros terminales",
        "Registros terminales con duración cero"
    ],
    "Valor": [
        total_records,
        unique_ids,
        duplicate_ids,
        missing_ids,
        missing_names,
        missing_wbs,
        zero_duration,
        missing_duration,
        int(df_wbs_audit["Has_Children"].sum()),
        int(df_wbs_audit["Is_Terminal"].sum()),
        int((
            df_wbs_audit["Is_Terminal"]
            & (pd.to_numeric(
                df_wbs_audit["Duration_Plan_Days"],
                errors="coerce"
            ) == 0)
        ).sum())
    ]
})

display(df_wbs_inventory)

# ------------------------------------------------------------
# 5. Identificar los registros terminales con duración cero
# ------------------------------------------------------------

df_zero_terminal = df_wbs_audit[
    df_wbs_audit["Is_Terminal"]
    & (
        pd.to_numeric(
            df_wbs_audit["Duration_Plan_Days"],
            errors="coerce"
        ) == 0
    )
].copy()

columns_to_show = [
    "ID",
    "Name",
    "WBS",
    "Duration_Plan_Days",
    "Has_Children"
]

display(
    df_zero_terminal[columns_to_show]
    .sort_values("ID")
    .reset_index(drop=True)
)

print("=" * 65)
print("AUDITORÍA INICIAL WBS")
print("=" * 65)
print(f"Registros totales: {total_records}")
print(f"Terminales con duración cero: {len(df_zero_terminal)}")
print("Archivos originales: SIN MODIFICAR")

la estructura básica está limpia.
- 167 registros y 167 identificadores únicos.
- Ningún ID, nombre, WBS o duración vacíos.
- 32 nodos con descendientes.
- 135 registros terminales.
- 28 terminales con duración cero.

In [15]:
# ============================================================
# AUDITORÍA WBS — 02. INSPECCIÓN DETALLADA
# ============================================================

# Mostrar todas las columnas originales de los 28 registros
original_columns = df_master.columns.tolist()

df_zero_detail = (
    df_master.loc[
        df_master["ID"].isin(df_zero_terminal["ID"]),
        original_columns
    ]
    .sort_values("ID")
    .reset_index(drop=True)
)

print("COLUMNAS DISPONIBLES:")
print(original_columns)

print("\nREGISTROS TERMINALES CON DURACIÓN CERO:")
display(df_zero_detail)

print("\nRESUMEN DE CAMPOS CON INFORMACIÓN:")
for col in original_columns:
    non_null = df_zero_detail[col].notna().sum()
    if non_null > 0:
        print(f"{col}: {non_null}/{len(df_zero_detail)}")

| Grupo | IDs | Interpretación provisional |
| :--- | :--- | :--- |
| **Permisos** | 3, 4, 5 | Hitos administrativos |
| **Pre-installation** | 52, 58, 64, 70, 76, 82, 88 | Eventos de preparación asociados a cada unidad |
| **WEC 1–7, grupo 1** | 110–116 | Eventos asociados a la secuencia de instalación |
| **WEC 1–7, grupo 2** | 142–148 | Eventos asociados a otra secuencia de ejecución |
| **Excavation for cabin** | 152 | Posible actividad con duración no especificada |
| **Energize** | 159 | Hito de puesta en tensión |
| **Order connection** | 162 | Hito administrativo o de conexión |
| **Finish** | 166 | Hito final |

Hay dos precauciones:
- Excavation for cabin parece una actividad ejecutable, pero no debemos asignarle duración sin evidencia.

- Pre installation y los registros WEC tienen relaciones SF y FS. No debemos simplificarlas ni reinterpretarlas sin comprobar cómo se utilizan en el CPM.

In [18]:
# ============================================================
# AUDITORÍA WBS — 03. RELACIONES Y FECHAS
# ============================================================

# 1. Vista completa de los campos relevantes
cols = [
    "ID",
    "Name",
    "WBS",
    "WBS_Level",
    "Predecessors",
    "Successors",
    "Baseline_Start",
    "Baseline_End",
    "Duration_Plan_Days"
]

df_relations = (
    df_zero_terminal[cols]
    .sort_values("ID")
    .reset_index(drop=True)
)

pd.set_option("display.max_rows", 40)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

display(df_relations)

# 2. Comprobar referencias a IDs existentes
valid_ids = set(df_master["ID"].dropna())

def extract_referenced_ids(value):
    if pd.isna(value):
        return []

    import re
    return [
        int(x)
        for x in re.findall(r"(?:FS|SF|SS|FF)(\d+)", str(value))
    ]

relation_checks = []

for _, row in df_relations.iterrows():
    for relation_type in ["Predecessors", "Successors"]:
        refs = extract_referenced_ids(row[relation_type])

        for ref_id in refs:
            relation_checks.append({
                "ID": row["ID"],
                "Name": row["Name"],
                "Relation_Field": relation_type,
                "Referenced_ID": ref_id,
                "Reference_Exists": ref_id in valid_ids
            })

df_reference_checks = pd.DataFrame(relation_checks)

if not df_reference_checks.empty:
    display(
        df_reference_checks[
            ~df_reference_checks["Reference_Exists"]
        ]
    )

    print(
        "Referencias examinadas:",
        len(df_reference_checks)
    )
    print(
        "Referencias inexistentes:",
        (~df_reference_checks["Reference_Exists"]).sum()
    )
else:
    print("No se han encontrado referencias explícitas.")

# 3. Resumen de fechas y duraciones
df_date_checks = df_relations.assign(
    Start=pd.to_datetime(df_relations["Baseline_Start"]),
    End=pd.to_datetime(df_relations["Baseline_End"])
)

df_date_checks["Same_Date"] = (
    df_date_checks["Start"] == df_date_checks["End"]
)

print("\nRESUMEN DE FECHAS:")
print(
    df_date_checks["Same_Date"]
    .value_counts(dropna=False)
)

print("\nAUDITORÍA DE RELACIONES COMPLETADA.")

| Registros | Interpretación |
| :--- | :--- |
| **3, 4 y 5 — permisos/RUP** | Hitos administrativos candidatos |
| **52, 58, 64, 70, 76, 82 y 88 — Pre installation** | Eventos de preparación ligados a actividades de instalación |
| **110–116 y 142–148 — WEC 1–7** | Hitos asociados a unidades de aerogenerador |
| **152 — Excavation for cabin** | Revisión necesaria: parece una actividad ejecutable, no un hito |
| **159 — Energize** | Hito de puesta en tensión |
| **162 — Order connection** | Hito administrativo o de solicitud |
| **166 — Finish** | Hito final |

In [21]:
# ============================================================
# AUDITORÍA WBS — 04. PARSER COMPLETO DE DEPENDENCIAS
# ============================================================

import re
import pandas as pd

def parse_dependencies(value):
    if pd.isna(value):
        return []

    pattern = r'(?:(FS|SF|SS|FF)\s*(\d+)|(\d+)\s*(FS|SF|SS|FF))'

    matches = re.findall(pattern, str(value).upper())

    dependencies = []

    for match in matches:
        if match[0]:  # Formato FS16
            relation_type = match[0]
            referenced_id = int(match[1])
        else:         # Formato 16FS
            referenced_id = int(match[2])
            relation_type = match[3]

        dependencies.append({
            "Relation_Type": relation_type,
            "Referenced_ID": referenced_id
        })

    return dependencies


valid_ids = set(df_master["ID"].dropna().astype(int))

relation_checks = []

for _, row in df_zero_terminal.iterrows():

    for field in ["Predecessors", "Successors"]:

        for dependency in parse_dependencies(row[field]):

            relation_checks.append({
                "ID": int(row["ID"]),
                "Name": row["Name"],
                "Relation_Field": field,
                "Relation_Type": dependency["Relation_Type"],
                "Referenced_ID": dependency["Referenced_ID"],
                "Reference_Exists": (
                    dependency["Referenced_ID"] in valid_ids
                )
            })

df_reference_checks = pd.DataFrame(relation_checks)

print("REFERENCIAS DETECTADAS:", len(df_reference_checks))

print("\nDISTRIBUCIÓN POR CAMPO:")
display(
    df_reference_checks["Relation_Field"]
    .value_counts()
    .rename_axis("Campo")
    .to_frame("Referencias")
)

print("\nDISTRIBUCIÓN POR TIPO:")
display(
    df_reference_checks["Relation_Type"]
    .value_counts()
    .rename_axis("Tipo")
    .to_frame("Referencias")
)

print("\nREFERENCIAS INEXISTENTES:")
df_invalid_references = df_reference_checks[
    ~df_reference_checks["Reference_Exists"]
]

display(df_invalid_references)

print(
    "Total de referencias inexistentes:",
    len(df_invalid_references)
)

print("\nREFERENCIAS DE LOS 28 REGISTROS:")
display(
    df_reference_checks.sort_values(
        ["ID", "Relation_Field", "Referenced_ID"]
    ).reset_index(drop=True)
)

57 referencias detectadas: 31 en predecesores y 26 en sucesores.



49 relaciones FS y 8 SF.



Cero referencias a IDs inexistentes.



El parser ha recuperado 31 referencias adicionales que la primera versión no detectaba.

In [23]:
# ============================================================
# AUDITORÍA WBS — 05. CLASIFICACIÓN PROVISIONAL
# ============================================================

milestone_ids = [
    3, 4, 5,
    110, 111, 112, 113, 114, 115, 116,
    142, 143, 144, 145, 146, 147, 148,
    159, 162, 166
]

preparation_ids = [
    52, 58, 64, 70, 76, 82, 88
]

review_ids = [152]

df_zero_terminal["Record_Type_Audit"] = "UNCLASSIFIED"

df_zero_terminal.loc[
    df_zero_terminal["ID"].isin(milestone_ids),
    "Record_Type_Audit"
] = "MILESTONE_CANDIDATE"

df_zero_terminal.loc[
    df_zero_terminal["ID"].isin(preparation_ids),
    "Record_Type_Audit"
] = "PREPARATION_EVENT_CANDIDATE"

df_zero_terminal.loc[
    df_zero_terminal["ID"].isin(review_ids),
    "Record_Type_Audit"
] = "ACTIVITY_DURATION_REVIEW"

df_zero_terminal["Audit_Comment"] = ""

df_zero_terminal.loc[
    df_zero_terminal["Record_Type_Audit"] == "MILESTONE_CANDIDATE",
    "Audit_Comment"
] = "Clasificación provisional por nombre y contexto de dependencias."

df_zero_terminal.loc[
    df_zero_terminal["Record_Type_Audit"] == "PREPARATION_EVENT_CANDIDATE",
    "Audit_Comment"
] = "Evento de preparación con duración cero; confirmar definición en origen."

df_zero_terminal.loc[
    df_zero_terminal["Record_Type_Audit"] == "ACTIVITY_DURATION_REVIEW",
    "Audit_Comment"
] = "Excavation for cabin parece trabajo ejecutable; revisar duración en origen."

# Resumen
print("CLASIFICACIÓN PROVISIONAL")
display(
    df_zero_terminal["Record_Type_Audit"]
    .value_counts()
    .rename_axis("Tipo")
    .to_frame("Registros")
)

# Detalle
display(
    df_zero_terminal[
        [
            "ID",
            "Name",
            "WBS",
            "Duration_Plan_Days",
            "Predecessors",
            "Successors",
            "Record_Type_Audit",
            "Audit_Comment"
        ]
    ].sort_values("ID").reset_index(drop=True)
)

print("\nRegistros clasificados:", 
      (df_zero_terminal["Record_Type_Audit"] != "UNCLASSIFIED").sum())

print("Duraciones originales modificadas: NO")

Perfecto. La clasificación queda cerrada: 28 de 28 registros revisados, distribuidos en:
- 20 candidatos a hito.
- 7 eventos de preparación candidatos.
- 1 actividad cuya duración debe revisarse en el documento de origen: ID 152 — Excavation for cabin.
No hay que corregir ahora ninguna duración ni dependencia. La clasificación es una anotación de auditoría, no una modificación del cronograma. También queda validado que las 57 referencias detectadas existen.

Con esto damos por terminada la revisión sustantiva de la normalización WBS. Solo falta guardar la evidencia para que no se pierda.

In [24]:
# ============================================================
# AUDITORÍA WBS — 06. EXPORTACIÓN
# ============================================================

from pathlib import Path
import pandas as pd

output_dir = Path(r"C:\ds4b\C2011-13 Wind Farm_v4\07_Outputs")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "WBS_Normalization_Audit.xlsx"

df_wbs_summary = pd.DataFrame({
    "Control": [
        "Registros totales",
        "IDs únicos",
        "Nodos con descendientes",
        "Registros terminales",
        "Registros terminales con duración cero",
        "Candidatos a hito",
        "Eventos de preparación candidatos",
        "Actividades pendientes de revisar",
        "Referencias de dependencias examinadas",
        "Referencias inexistentes"
    ],
    "Resultado": [
        len(df_master),
        df_master["ID"].nunique(),
        int(df_wbs_audit["Has_Children"].sum()),
        int(df_wbs_audit["Is_Terminal"].sum()),
        len(df_zero_terminal),
        int((df_zero_terminal["Record_Type_Audit"] == "MILESTONE_CANDIDATE").sum()),
        int((df_zero_terminal["Record_Type_Audit"] == "PREPARATION_EVENT_CANDIDATE").sum()),
        int((df_zero_terminal["Record_Type_Audit"] == "ACTIVITY_DURATION_REVIEW").sum()),
        len(df_reference_checks),
        len(df_invalid_references)
    ]
})

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    df_wbs_summary.to_excel(
        writer, sheet_name="Audit Summary", index=False
    )

    df_wbs_audit.to_excel(
        writer, sheet_name="WBS Hierarchy", index=False
    )

    df_zero_terminal.to_excel(
        writer, sheet_name="Zero Duration Review", index=False
    )

    df_reference_checks.to_excel(
        writer, sheet_name="Dependency References", index=False
    )

print("AUDITORÍA WBS EXPORTADA")
print(output_file)
print(f"Tamaño: {output_file.stat().st_size / 1024:.2f} KB")
print("Hojas:", pd.ExcelFile(output_file).sheet_names)

In [25]:
# ============================================================
# INTEGRATION QA — 01. INVENTARIO DE ENTREGABLES
# ============================================================

from pathlib import Path
import pandas as pd

root = Path(r"C:\ds4b\C2011-13 Wind Farm_v4")

files_to_check = {
    "Master con materiales": root / "05_Outputs" / "Activity_Master_with_Materials.xlsx",
    "Registro de seguimiento": root / "07_Outputs" / "Activity_Tracking_Register.xlsx",
    "Control del calendario": root / "07_Outputs" / "Schedule_Control.xlsx",
    "Configuración de escenarios": root / "07_Outputs" / "Scenario_Configuration.xlsx",
    "Simulación de escenarios": root / "07_Outputs" / "Scenario_Simulation_Results.xlsx",
    "Evaluación de acciones correctivas": root / "07_Outputs" / "Corrective_Action_Evaluation.xlsx",
    "Evaluación económica": root / "07_Outputs" / "Economic_Impact_Assessment.xlsx",
    "Auditoría WBS": root / "07_Outputs" / "WBS_Normalization_Audit.xlsx"
}

inventory = []

for name, path in files_to_check.items():

    exists = path.exists()

    if exists:
        excel = pd.ExcelFile(path)

        inventory.append({
            "Entregable": name,
            "Existe": True,
            "Tamaño_KB": round(path.stat().st_size / 1024, 2),
            "Hojas": " | ".join(excel.sheet_names)
        })

    else:
        inventory.append({
            "Entregable": name,
            "Existe": False,
            "Tamaño_KB": None,
            "Hojas": "NO ENCONTRADO"
        })

df_integration_inventory = pd.DataFrame(inventory)

display(df_integration_inventory)

print("\nARCHIVOS ENCONTRADOS:",
      df_integration_inventory["Existe"].sum(),
      "de", len(df_integration_inventory))

print("ARCHIVOS AUSENTES:",
      (~df_integration_inventory["Existe"]).sum())

In [26]:
# ============================================================
# INTEGRATION QA — 02. LECTURA DE RESULTADOS
# ============================================================

from pathlib import Path
import pandas as pd

out = Path(r"C:\ds4b\C2011-13 Wind Farm_v4\07_Outputs")

sheets_to_read = {
    "Schedule_Control.xlsx": ["Schedule Control"],
    "Scenario_Configuration.xlsx": ["Scenarios"],
    "Scenario_Simulation_Results.xlsx": ["Scenario Summary"],
    "Corrective_Action_Evaluation.xlsx": [
        "Decision Summary",
        "Corrective Actions",
        "Supply Risk"
    ],
    "Economic_Impact_Assessment.xlsx": [
        "Executive Summary",
        "Combined SC03"
    ]
}

qa_data = {}

for filename, sheet_names in sheets_to_read.items():

    path = out / filename

    for sheet in sheet_names:

        df = pd.read_excel(path, sheet_name=sheet)
        key = f"{filename} | {sheet}"
        qa_data[key] = df

        print(f"\n{'='*75}")
        print(key)
        print("Dimensiones:", df.shape)
        print("Columnas:", list(df.columns))

        # Las hojas resumen suelen ser pequeñas.
        # Si alguna es extensa, mostramos solo las primeras filas.
        display(df.head(10))

print("\nLECTURA DE RESULTADOS COMPLETADA.")

Los resultados que has pegado muestran una coherencia numérica clara entre simulación y acciones correctivas:
- SC01: 299 días antes y después; el impacto de suministro baja de 15 a 5 días.
- SC02: 306 → 301 días; se recuperan 5 de los 7 días de retraso.
- SC03: 306 → 301 días; se recuperan 5 días y el impacto de suministro queda en 5.
- La acción CA01 no elimina el riesgo de suministro: el retraso total residual es de 40 días frente a un buffer de 21; siguen excediéndose en 19 días.
- En el escenario económico realista, el beneficio neto combinado estimado es 13.900 €, con ROI del 95,21 %. Son resultados de supuestos ilustrativos, no ahorros observados.
Hay una distinción importante: la mitigación de suministro y la recuperación del calendario son dos efectos diferentes. Los entregables los mantienen separados correctamente. Vamos a automatizar estas comprobaciones y revisar también la aritmética económica

In [29]:
# ============================================================
# INTEGRATION QA — 03. VALIDACIÓN CRUZADA
# ============================================================

import pandas as pd
import numpy as np

# Recuperar hojas ya leídas
df_scen = qa_data[
    "Scenario_Simulation_Results.xlsx | Scenario Summary"
].copy()

df_dec = qa_data[
    "Corrective_Action_Evaluation.xlsx | Decision Summary"
].copy()

df_actions = qa_data[
    "Corrective_Action_Evaluation.xlsx | Corrective Actions"
].copy()

df_supply = qa_data[
    "Corrective_Action_Evaluation.xlsx | Supply Risk"
].copy()

df_econ = qa_data[
    "Economic_Impact_Assessment.xlsx | Executive Summary"
].copy()

df_combined = qa_data[
    "Economic_Impact_Assessment.xlsx | Combined SC03"
].copy()


checks = []

def add_check(name, condition, detail):
    checks.append({
        "Control": name,
        "Status": "PASS" if bool(condition) else "FAIL",
        "Detail": detail
    })


# ------------------------------------------------------------
# 1. SIMULACIÓN: DURACIONES Y RETRASOS
# ------------------------------------------------------------

expected_scenarios = {
    "SC01": (299, 0, 0, 15),
    "SC02": (306, 7, 7, 0),
    "SC03": (306, 7, 7, 15)
}

for scenario, expected in expected_scenarios.items():

    row = df_scen[df_scen["Scenario_ID"] == scenario]

    if len(row) == 1:
        r = row.iloc[0]

        actual = (
            r["Project_Duration_Days"],
            r["Project_Delay_Days"],
            r["Activity_Delay_Days"],
            r["Supply_Delay_Days"]
        )

        add_check(
            f"Simulación {scenario}",
            tuple(actual) == expected,
            f"Esperado={expected}; obtenido={tuple(actual)}"
        )
    else:
        add_check(
            f"Simulación {scenario}",
            False,
            f"Número de registros encontrados: {len(row)}"
        )


# ------------------------------------------------------------
# 2. ACCIONES CORRECTIVAS: RECUPERACIÓN
# ------------------------------------------------------------

expected_recovery = {
    "SC01": (299, 299, 0, 5),
    "SC02": (306, 301, 5, 0),
    "SC03": (306, 301, 5, 5)
}

for scenario, expected in expected_recovery.items():

    row = df_dec[df_dec["Scenario_ID"] == scenario]

    if len(row) == 1:
        r = row.iloc[0]

        actual = (
            r["Duration_Before_Actions"],
            r["Duration_After_Actions"],
            r["Project_Days_Recovered"],
            r["Supply_Impact_After"]
        )

        add_check(
            f"Acciones correctivas {scenario}",
            tuple(actual) == expected,
            f"Esperado={expected}; obtenido={tuple(actual)}"
        )
    else:
        add_check(
            f"Acciones correctivas {scenario}",
            False,
            f"Número de registros encontrados: {len(row)}"
        )


# ------------------------------------------------------------
# 3. ACCIONES INDIVIDUALES
# ------------------------------------------------------------

ca01 = df_actions[df_actions["Action_ID"] == "CA01"]
ca02 = df_actions[df_actions["Action_ID"] == "CA02"]

if len(ca01) == 1:
    r = ca01.iloc[0]
    add_check(
        "CA01: mitigación de suministro",
        (
            r["Scenario_Impact_Before_Days"] == 15
            and r["Scenario_Impact_After_Days"] == 5
            and r["Recovered_Days"] == 10
        ),
        "Impacto 15 → 5 días; recuperación esperada: 10 días."
    )
else:
    add_check("CA01: mitigación de suministro", False,
              f"Registros encontrados: {len(ca01)}")

if len(ca02) == 1:
    r = ca02.iloc[0]
    add_check(
        "CA02: recuperación de actividad",
        (
            r["Scenario_Impact_Before_Days"] == 7
            and r["Scenario_Impact_After_Days"] == 2
            and r["Recovered_Days"] == 5
        ),
        "Impacto 7 → 2 días; recuperación esperada: 5 días."
    )
else:
    add_check("CA02: recuperación de actividad", False,
              f"Registros encontrados: {len(ca02)}")


# ------------------------------------------------------------
# 4. RIESGO DE SUMINISTRO RESIDUAL
# ------------------------------------------------------------

supply_ok = (
    (df_supply["Baseline_Supply_Delay_Days"] == 35).all()
    and (df_supply["Buffer_Days"] == 21).all()
    and (df_supply["Residual_Total_Supply_Delay_Days"] == 40).all()
    and (df_supply["Residual_Buffer_Excess_Days"] == 19).all()
    and (df_supply["Supply_Risk_Reduction_Days"] == 10).all()
)

add_check(
    "Riesgo residual de suministro",
    supply_ok,
    "Retraso base 35 + impacto residual 5 = 40 días; "
    "buffer 21; exceso residual 19 días."
)


# ------------------------------------------------------------
# 5. COHERENCIA ECONÓMICA
# ------------------------------------------------------------

econ_arithmetic_ok = np.allclose(
    df_econ["Combined_Net_Benefit_EUR"],
    (
        df_econ["Combined_Avoided_Cost_EUR"]
        - df_econ["Combined_Intervention_Cost_EUR"]
    ),
    atol=0.01
)

add_check(
    "Aritmética del beneficio combinado",
    econ_arithmetic_ok,
    "Beneficio neto = costes evitados - coste de intervención."
)

econ_roi_ok = np.allclose(
    df_econ["Combined_ROI_Pct"],
    (
        df_econ["Combined_Net_Benefit_EUR"]
        / df_econ["Combined_Intervention_Cost_EUR"]
        * 100
    ),
    atol=0.02
)

add_check(
    "Cálculo del ROI combinado",
    econ_roi_ok,
    "ROI = beneficio neto / coste de intervención × 100."
)


# ------------------------------------------------------------
# 6. COHERENCIA ENTRE RESUMEN ECONÓMICO Y SC03
# ------------------------------------------------------------

econ_realistic = df_econ[
    df_econ["Scenario"] == "Realistic"
]

combined_realistic = df_combined[
    df_combined["Scenario"] == "Realistic"
]

if len(econ_realistic) == 1 and len(combined_realistic) == 1:

    e = econ_realistic.iloc[0]
    c = combined_realistic.iloc[0]

    consistency = (
        np.isclose(
            e["Combined_Net_Benefit_EUR"],
            c["Net_Benefit_EUR"]
        )
        and np.isclose(
            e["Combined_ROI_Pct"],
            c["ROI_Pct"]
        )
    )

    add_check(
        "Consistencia entre resúmenes económicos",
        consistency,
        "Escenario realista: Executive Summary frente a Combined SC03."
    )
else:
    add_check(
        "Consistencia entre resúmenes económicos",
        False,
        "No se ha encontrado un único registro Realistic en ambas hojas."
    )


# ------------------------------------------------------------
# RESULTADO FINAL
# ------------------------------------------------------------

df_integration_checks = pd.DataFrame(checks)

display(df_integration_checks)

passed = (df_integration_checks["Status"] == "PASS").sum()
failed = (df_integration_checks["Status"] == "FAIL").sum()

print("\n========== RESULTADO DE INTEGRATION QA ==========")
print(f"Controles superados: {passed}")
print(f"Controles fallidos:  {failed}")
print(f"Total de controles:  {len(df_integration_checks)}")

if failed == 0:
    print("\nRESULTADO: VALIDACIÓN CRUZADA SUPERADA.")
else:
    print("\nRESULTADO: HAY CONTROLES QUE REQUIEREN REVISIÓN.")

La integración numérica está validada:
- Los tres escenarios conservan los resultados esperados.
- Las dos acciones correctivas producen las recuperaciones previstas.
- El riesgo residual de suministro está correctamente separado del retraso del proyecto.
- Beneficio neto y ROI cuadran aritméticamente.
- Los dos resúmenes económicos son coherentes.

In [30]:
# ============================================================
# INTEGRATION QA — 04. EXPORTACIÓN FINAL
# ============================================================

from pathlib import Path
import pandas as pd

output_dir = Path(
    r"C:\ds4b\C2011-13 Wind Farm_v4\07_Outputs"
)

output_file = output_dir / "Project_Integration_QA.xlsx"

df_qa_summary = pd.DataFrame({
    "Indicador": [
        "Entregables encontrados",
        "Entregables esperados",
        "Controles ejecutados",
        "Controles superados",
        "Controles fallidos",
        "Resultado"
    ],
    "Valor": [
        int(df_integration_inventory["Existe"].sum()),
        len(df_integration_inventory),
        len(df_integration_checks),
        int((df_integration_checks["Status"] == "PASS").sum()),
        int((df_integration_checks["Status"] == "FAIL").sum()),
        "PASS" if (df_integration_checks["Status"] == "PASS").all()
        else "REVIEW REQUIRED"
    ]
})

df_qa_scope = pd.DataFrame({
    "Aspecto": [
        "Alcance",
        "Método",
        "Datos originales",
        "Modelos y simulaciones",
        "Limitación económica",
        "Limitación de suministro"
    ],
    "Descripción": [
        "Comprobación documental y numérica de los entregables existentes.",
        "Lectura de archivos y validación cruzada de resultados.",
        "No modificados.",
        "No recalculados ni reejecutados.",
        "Los costes evitados y ROI se basan en supuestos ilustrativos.",
        "La mitigación del impacto de escenario no elimina el retraso base."
    ]
})

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    df_qa_summary.to_excel(
        writer, sheet_name="QA Summary", index=False
    )

    df_integration_inventory.to_excel(
        writer, sheet_name="Deliverable Inventory", index=False
    )

    df_integration_checks.to_excel(
        writer, sheet_name="Cross-Validation", index=False
    )

    df_qa_scope.to_excel(
        writer, sheet_name="Scope and Limitations", index=False
    )

print("INTEGRATION QA EXPORTADO")
print(output_file)
print(f"Tamaño: {output_file.stat().st_size / 1024:.2f} KB")
print("Hojas:", pd.ExcelFile(output_file).sheet_names)